In [ ]:
import logging
import tqdm
import random
from exp.run import ExperimentRun
from exp.config import TransformerExperiments, CNNExperiments, LargeTransformerExperiments
logging.basicConfig(level=logging.ERROR)

Three variable should be taken care of:
1. `conf_index`: an index for configureation (1 for CNN, 2 for Transformer, 3 for Qwen3 and Pythia)
2. `total_run`: a number of runs
3. `gpu_ids`: keep it as it if you have more than two GPUs in your environment.

In [ ]:
total_run: int = 1300
gpu_ids = [0, 1]

In [ ]:
configures = [
	CNNExperiments(),
	TransformerExperiments(),
	LargeTransformerExperiments()
]
exp_instances = []
for config in configures:
	config.repeats = 1
	config.gpu_id = 0
	config.result_verification = True
	exp_instances.append(ExperimentRun(config))

In [ ]:
print(f"Preparing {total_run} test configuration...")
for i in tqdm.tqdm(range(total_run)):
	instance = random.choice(exp_instances)
	instance.prepare_monte_carlo_experiments_data(number=1, gpus=gpu_ids)

In [ ]:
print(f"Start Monte Carlo experiments...")
for instance in exp_instances:
	if isinstance(instance._config, LargeTransformerExperiments):
		print(f"[C]Execute {instance._config.run_id}'s experiment...")
		instance.run_experiments_solving_compatibility_issue()
	else:
		print(f"Execute {instance._config.run_id}'s experiment...")
		instance.run_experiments()

	instance.to_evaluation_result()



# Visualize the Results

In [ ]:
from pathlib import Path
from plot.plot import ExperimentPlot, DataAggregation
e_plot = ExperimentPlot()


In [ ]:
da = DataAggregation(random_dir=False)
result_dir = da.aggregate(
	Path().home().joinpath(exp_instances[0]._config.run_id),
	Path().home().joinpath(exp_instances[1]._config.run_id),
	Path().home().joinpath(exp_instances[2]._config.run_id),
	subdir_name="MonteCarlo"
)
print(f"All results will be copied to {result_dir}")


## MER

In [ ]:
filename = "ViewOnly-Mento Carlo-MER-CNN"
fig = e_plot.plot_relative_error_in_box_diagram_with_verification_data(
    title=filename,
    data_dir=result_dir.joinpath("CNN"),
    overall_median=False,
    view_mode = True,
)
fig.show()

In [ ]:
filename = "ViewOnly-Mento Carlo-MER-Transformer"
fig = e_plot.plot_relative_error_in_box_diagram_with_verification_data(
    title=filename,
    data_dir=result_dir.joinpath("Transformer"),
    overall_median=False,
    view_mode = True,
)
fig.show()

## PEF

In [ ]:
filename = "ViewOnly-Mento Carlo-PEF"
fig = e_plot.plot_probability_estimation_vs_error_scatter_diagram_model_base(
    title=filename,
    data_dir=result_dir,
    view_mode=True
)
fig.show()